# Single-task text classification with EgyBERT, Phase 2

I build and evaluate single-task text classification models using EgyBERT (faisalq/EgyBERT) on the SVR Voicemail Corpus. I train one model per field on the train split (3,575 rows), evaluate on the validation split (1,003 rows), and record validation macro-F1 across 5 fixed seeds. I score customer_state on 4 classes and record the single unclear validation row under exceptions. I compare the results against the TF-IDF baseline.

## Contents
1. Setup, configuration, and shared helpers
2. Load clean training tables and class weights
3. Tokenizer verification and sequence lengths
4. Training pipeline and prediction files
5. Evaluation with the infrastructure harness
6. Comparison against the TF-IDF baseline

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch

from infra.shared_config import load_shared_config
from infra.experiment_config import ExperimentConfig
from infra.predictions_file import predictions_path, read_predictions_file
from infra.harness import evaluate_predictions
from infra.run_record import load_run_record

# Paths
REPO_ROOT = Path(".").resolve()
SHARED_CONFIG_PATH = REPO_ROOT / "notebooks" / "training" / "shared_config.json"
DATA_DIR = REPO_ROOT.parent / "svr-data" / "final"
RUNS_DIR = REPO_ROOT / "runs"

shared_config = load_shared_config(SHARED_CONFIG_PATH)
print("Shared config loaded successfully:")
for k, v in shared_config.items():
    print(f"  {k}: {v}")

Shared config loaded successfully:
  split_seed: 42
  experiment_seeds: [42, 1337, 2023, 7, 99]
  max_seq_len: 128
  split_ratio: 70/20/10
  dataset_version: v2
  dataset: svr-voicemail-corpus
  train_rows: 3575
  val_rows: 1003
  test_rows: 518


## 2. Load clean training tables and class weights

I load the train and validation tables from the data worktree. The split is frozen at 3,575 train rows and 1,003 validation rows. I use text_norm as the model input. Class weights come from the train split only and are loaded from class_weights.json.

In [2]:
train_df = pd.read_csv(DATA_DIR / "clean_training_train.csv")
val_df = pd.read_csv(DATA_DIR / "clean_training_val.csv")

print(f"Train rows: {len(train_df)} (expected: {shared_config['train_rows']})  [OK]")
print(f"Val rows:   {len(val_df)} (expected: {shared_config['val_rows']})    [OK]")

weights_data = json.loads((DATA_DIR / "class_weights.json").read_text(encoding="utf-8"))
classes_by_field = {
    field: list(weights.keys())
    for field, weights in weights_data["weights"].items()
}

print("\nFields and classes:")
for f, cls in classes_by_field.items():
    print(f"  {f} ({len(cls)} classes): {cls}")

Train rows: 3575 (expected: 3575)  [OK]
Val rows:   1003 (expected: 1003)    [OK]

Fields and classes:
  complaint_category (11 classes): ['app_and_account', 'balance_and_data', 'billing_and_charges', 'branch_and_staff', 'cancellation_request', 'network_and_connectivity', 'no_reply_or_long_wait', 'offers_and_packages', 'other', 'sim_services', 'thanks_or_praise']
  complaint_urgency (3 classes): ['high', 'low', 'medium']
  customer_state (4 classes): ['calm', 'frustrated', 'mad', 'satisfied']
  churn_risk (2 classes): ['at_risk', 'not_at_risk']


## 3. Tokenizer verification and sequence lengths

I verify the EgyBERT tokenizer on the normalized text. I confirm that max sequence length 128 covers all transcripts in the dataset without truncation.

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("faisalq/EgyBERT")
train_lengths = [len(tokenizer.encode(str(t), truncation=False)) for t in train_df["text_norm"]]
val_lengths = [len(tokenizer.encode(str(t), truncation=False)) for t in val_df["text_norm"]]

all_lengths = train_lengths + val_lengths
p50 = pd.Series(all_lengths).quantile(0.50)
p90 = pd.Series(all_lengths).quantile(0.90)
p99 = pd.Series(all_lengths).quantile(0.99)
max_len = max(all_lengths)

print(f"Token lengths (train + val, {len(all_lengths)} rows) -> p50: {p50:.1f}, p90: {p90:.1f}, p99: {p99:.1f}, max: {max_len}")
assert max_len <= shared_config["max_seq_len"]
print(f"Max sequence length 128 fully covers the dataset (max = {max_len})  [OK]")

Token lengths (train + val, 4578 rows) -> p50: 19.0, p90: 30.0, p99: 50.0, max: 88
Max sequence length 128 fully covers the dataset (max = 88)  [OK]


## 4. Training pipeline and prediction files

I run single-task fine-tuning for all 4 fields across the 5 seeds: 42, 1337, 2023, 7, 99. The hyperparameters follow the model selection report: learning rate 3e-5, effective batch size 16, epochs up to 5 with early stopping patience 2, warmup ratio 0.1, weight decay 0.01, and AdamW optimizer. For each seed, prediction files are written in Aya's infra format.

In [4]:
seeds = shared_config["experiment_seeds"]
print(f"Experiment seeds in order: {seeds}")

# Verify predictions files existence
for s in seeds:
    pred_file = predictions_path(RUNS_DIR, "001", "val", s)
    exists = pred_file.exists()
    print(f"  Seed {s:<4}: {pred_file.name} (exists: {exists})")

Experiment seeds in order: [42, 1337, 2023, 7, 99]
  Seed 42  : predictions_val_seed42.json (exists: True)
  Seed 1337: predictions_val_seed1337.json (exists: True)
  Seed 2023: predictions_val_seed2023.json (exists: True)
  Seed 7   : predictions_val_seed7.json (exists: True)
  Seed 99  : predictions_val_seed99.json (exists: True)


## 5. Evaluation with the infrastructure harness

I evaluate each seed's predictions file using the infrastructure harness. The customer_state metric is scored on the 4 classes present in train (calm, frustrated, mad, satisfied), and the single validation row with true label unclear is excluded and documented in exceptions.

In [5]:
UNCLEAR_REASON = (
    "unclear has 0 rows in train (1 in val, 0 in test), so the model never trains on it; "
    "excluded from customer_state macro-F1 only (Hazem's decision)."
)
EXCLUSIONS = {
    "customer_state": {
        "labels": ["unclear"],
        "reason": UNCLEAR_REASON,
    }
}

truth_rows = [
    {
        "audio_file": row["audio_file"],
        "complaint_category": row["complaint_category"],
        "complaint_urgency": row["complaint_urgency"],
        "customer_state": row["customer_state"],
        "churn_risk": row["churn_risk"],
    }
    for _, row in val_df.iterrows()
]

seed_results = []
for s in seeds:
    pred_file = predictions_path(RUNS_DIR, "001", "val", s)
    pred_data = read_predictions_file(pred_file, classes_by_field, expected_split="val")
    harness_rep = evaluate_predictions(
        split="val",
        truth_rows=truth_rows,
        pred_rows=pred_data["harness_rows"],
        classes_by_field=classes_by_field,
        exclusions_by_field=EXCLUSIONS,
    )
    metrics = {f: harness_rep["fields"][f]["macro"]["f1"] for f in classes_by_field}
    metrics["combined_mean"] = harness_rep["combined_macro_f1"]["mean"]
    seed_results.append({"seed": s, **metrics})

results_df = pd.DataFrame(seed_results)
print("Validation macro-F1 across seeds:")
print(results_df.to_string(index=False))

Validation macro-F1 across seeds:
 seed  complaint_category  complaint_urgency  customer_state  churn_risk  combined_mean
   42            0.745532           0.728561        0.740503    0.916165       0.782690
 1337            0.756854           0.701487        0.748308    0.916165       0.780703
 2023            0.722027           0.728955        0.741822    0.916165       0.777242
    7            0.771055           0.728397        0.748246    0.916165       0.790966
   99            0.761239           0.712605        0.740132    0.916165       0.782535


## 6. Comparison against the TF-IDF baseline

I load the run record from runs/exp_001/run_record.json and compare the aggregate metrics (mean and sample standard deviation with ddof=1) against the TF-IDF baseline on the corrected text.

In [6]:
record = load_run_record(RUNS_DIR / "exp_001" / "run_record.json")

baseline = {
    "complaint_category": 0.7605,
    "complaint_urgency": 0.6902,
    "customer_state": 0.7000,
    "churn_risk": 0.8629,
    "combined_mean": 0.7534,
}

rows = []
for f in ["complaint_category", "complaint_urgency", "customer_state", "churn_risk"]:
    egy_mean = record["aggregate"][f"{f}_macro_f1"]["mean"]
    egy_std = record["aggregate"][f"{f}_macro_f1"]["std"]
    base_val = baseline[f]
    diff = egy_mean - base_val
    status = "BEATS BASELINE" if diff > 0 else "BELOW BASELINE"
    rows.append({
        "field": f,
        "tfidf_baseline": f"{base_val:.4f}",
        "egybert_mean": f"{egy_mean:.4f}",
        "egybert_std": f"{egy_std:.4f}",
        "diff": f"{diff:+.4f}",
        "status": status,
    })

comb_base = baseline["combined_mean"]
comb_egy = float(np.mean([record["aggregate"][f"{f}_macro_f1"]["mean"] for f in ["complaint_category", "complaint_urgency", "customer_state", "churn_risk"]]))
comb_diff = comb_egy - comb_base
rows.append({
    "field": "combined_mean",
    "tfidf_baseline": f"{comb_base:.4f}",
    "egybert_mean": f"{comb_egy:.4f}",
    "egybert_std": "-",
    "diff": f"{comb_diff:+.4f}",
    "status": "BEATS BASELINE" if comb_diff > 0 else "BELOW BASELINE",
})

comparison_df = pd.DataFrame(rows)
print("\nSingle-task EgyBERT validation performance vs TF-IDF baseline:")
print(comparison_df.to_string(index=False))

print("\nSummary:")
print(f"EgyBERT beats the baseline on complaint_urgency ({comparison_df.loc[1, 'diff']})")
print(f"EgyBERT beats the baseline on customer_state ({comparison_df.loc[2, 'diff']})")
print(f"EgyBERT beats the baseline on churn_risk ({comparison_df.loc[3, 'diff']})")
print(f"EgyBERT beats the combined baseline by {comb_diff:+.4f} points ({comb_egy:.4f} vs {comb_base:.4f})")


Single-task EgyBERT validation performance vs TF-IDF baseline:
             field tfidf_baseline egybert_mean egybert_std    diff         status
complaint_category         0.7605       0.7513      0.0188 -0.0092 BELOW BASELINE
 complaint_urgency         0.6902       0.7200      0.0125 +0.0298 BEATS BASELINE
    customer_state         0.7000       0.7438      0.0041 +0.0438 BEATS BASELINE
        churn_risk         0.8629       0.9162      0.0000 +0.0533 BEATS BASELINE
     combined_mean         0.7534       0.7828           - +0.0294 BEATS BASELINE

Summary:
EgyBERT beats the baseline on complaint_urgency (+0.0298)
EgyBERT beats the baseline on customer_state (+0.0438)
EgyBERT beats the baseline on churn_risk (+0.0533)
EgyBERT beats the combined baseline by +0.0294 points (0.7828 vs 0.7534)
